In [ ]:
import torch
import numpy as np
import cv2
import os
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import torch.optim as optim
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

c:\Users\40757\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


![U-Net Architecture](./U-net_Arch.png)

In [ ]:
# The training data contains:

# img.png, label, RLE_encoding 

# EX: 
# 0007a71bf.jpg, 3, 18661 28 18863 82 19091 110 19347 110 19603
# Starting at pixel 18661, mask lasts 28 pixels 
# Starting at pixel 18863, mask lasts 82 pixels

def RLE_to_mask(rle, shape=(256, 1600)):
    # Makes 2 lists: Mask starting pixels and Mask end pixels 
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths

    # creates a 0-image
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    
    # covers the mask 
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    # Its F because the pixels in rle go COLUMN wise
    return img.reshape(shape, order='F')

In [ ]:
# Store [Img.png (3,256,1600), and a tensor masks (4,256,1600)]
class SteelDataset(Dataset):
    def __init__(self, df, img_folder):
        self.img_folder = img_folder
        self.grouped = df.groupby('ImageId')   
        self.image_ids = list(self.grouped.groups.keys())  # list of unique img id's

    def __len__(self):
        return len(self.image_ids)  

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        rows = self.grouped.get_group(image_id)     # get RLE encoding

        image = cv2.imread(f"{self.img_folder}/{image_id}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # read image

        mask = np.zeros((4, 256, 1600), dtype=np.float32) # create 4 empty masks

        for _, row in rows.iterrows():  # itterates through rle's (you can have multiple rows with the same image id)
            if pd.notna(row['EncodedPixels']):
                class_id = int(row['ClassId']) - 1
                mask[class_id] = RLE_to_mask(row['EncodedPixels']) # applies mask 

        # Convert to tensor
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).float()

        return image, mask

In [ ]:
df = pd.read_csv("../data/kaggle_data/train.csv")
df.columns = ["ImageId", "ClassId", "EncodedPixels"]

steel_data = SteelDataset(df,"../data/kaggle_data/train_images/")
train_loader = DataLoader(steel_data, batch_size=4, shuffle=True)

u_net = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",    
    in_channels=3,                  
    classes=4,                     
    activation= None          
)

optimizer = optim.AdamW(u_net.parameters(), lr=1e-4, weight_decay=1e-2)
loss_fn = smp.losses.DiceLoss(mode='multilabel')

In [ ]:
def train_fn(epochs, train_loader, net, optimizer):
    net.to(device) 
    
    for epoch in range(epochs):
        net.train()
        epoch_loss = 0
        
        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            outputs = net(images)   # me get output

            loss = loss_fn(outputs, targets) # me get loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

        print(f"Epoch {epoch} | Avg Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
train_fn(70, train_loader, u_net, optimizer)
torch.save(u_net.state_dict(), 'u-net_model.pth')

KeyboardInterrupt: 